# Module 5: Defense Sweeps

This staged notebook owns the longer malicious-fraction, Krum, and non-IID stress sweeps.


## Stage Goal

Stress-test defense behavior after the fixed-attack comparison is working. These sections rerun multiple FL jobs, so keep individual toggles off unless the compute budget is available.


## 1. Notebook Setup

Load shared helpers and make imports independent of notebook launch directory.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
MODULE_DIR = cwd if cwd.name == "5_Defensive_FL" else cwd / "5_Defensive_FL"
MODULE4_DIR = MODULE_DIR.parent / "4_Adversarial_FL"
MODULE4_SRC_DIR = MODULE4_DIR / "src"
REPO_ROOT = MODULE_DIR.parent

for path in (REPO_ROOT, MODULE4_DIR, MODULE4_SRC_DIR, MODULE_DIR):
    path_str = str(path.resolve())
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from notebook_utils import (
    load_context,
    record_config_snapshot,
    completed_rows,
    run_krum_hyperparameter_sweep,
    run_malicious_fraction_sweep,
    run_non_iid_stress,
    validate_artifacts,
)


## 2. Configuration and Sweep Controls

Each toggle controls one expensive section. The config owns the grids and defense lists.


In [ ]:
context = load_context("defense_sweeps_config.yaml", stage_name="defense_sweeps")
config_snapshot_path = record_config_snapshot(context)
sweep_controls = context.config.get("defense_sweeps", {})

RUN_MALICIOUS_FRACTION_SWEEP = bool(
    sweep_controls.get("run_malicious_fraction_sweep", True)
)
RUN_KRUM_HYPERPARAMETER_SWEEP = bool(
    sweep_controls.get("run_krum_hyperparameter_sweep", True)
)
RUN_NON_IID_STRESS = bool(
    sweep_controls.get("run_non_iid_stress", True)
)

{
    "malicious_fraction_sweep": RUN_MALICIOUS_FRACTION_SWEEP,
    "krum_hyperparameter_sweep": RUN_KRUM_HYPERPARAMETER_SWEEP,
    "non_iid_stress": RUN_NON_IID_STRESS,
}


## 3. Malicious-Fraction Sweep

Vary the malicious-client fraction while keeping the attack recipe and defense settings fixed.


In [ ]:
malicious_fraction_rows = (
    run_malicious_fraction_sweep(context)
    if RUN_MALICIOUS_FRACTION_SWEEP
    else []
)
malicious_fraction_rows[:3]


## 4. Krum Hyperparameter Sweep

Vary `byzantine_f`; infeasible settings are recorded as skipped rows.


In [ ]:
krum_sweep_rows = (
    run_krum_hyperparameter_sweep(context)
    if RUN_KRUM_HYPERPARAMETER_SWEEP
    else []
)
krum_sweep_rows


## 5. Non-IID Stress Test

Vary honest-client label skew and compare the configured stress-test defenses.


In [ ]:
non_iid_rows = (
    run_non_iid_stress(context)
    if RUN_NON_IID_STRESS
    else []
)
non_iid_rows[:3]


## Handoff Artifacts

Validate the artifacts produced by the enabled sweep sections.


In [ ]:
expected_artifacts = [config_snapshot_path]

if malicious_fraction_rows:
    expected_artifacts.append(context.artifact_path("module5_malicious_fraction_sweep.json"))
    if completed_rows(malicious_fraction_rows):
        expected_artifacts.extend([
            context.artifact_path("module5_malicious_fraction_accuracy.png"),
            context.artifact_path("module5_malicious_fraction_global_target_label_asr.png"),
        ])

if krum_sweep_rows:
    expected_artifacts.append(context.artifact_path("module5_krum_byzantine_f_sweep.json"))
    if completed_rows(krum_sweep_rows):
        expected_artifacts.extend([
            context.artifact_path("module5_krum_byzantine_f_accuracy.png"),
            context.artifact_path("module5_krum_byzantine_f_global_target_label_asr.png"),
        ])

if non_iid_rows:
    expected_artifacts.append(context.artifact_path("module5_non_iid_defense_stress.json"))
    if completed_rows(non_iid_rows):
        expected_artifacts.extend([
            context.artifact_path("module5_non_iid_accuracy.png"),
            context.artifact_path("module5_non_iid_global_target_label_asr.png"),
        ])

validate_artifacts(expected_artifacts)
